In [2]:

# CS 340 Project Two - Grazioso Salvare Dashboard

import base64
from pathlib import Path

import dash_leaflet as dl
import pandas as pd
import plotly.express as px
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash

from CRUD_Python_Module import AnimalShelter

JupyterDash.infer_jupyter_proxy_config()


###########################
# Data Manipulation / Model
###########################

# MongoDB login.
username = "aacuser"
password = "kelavi99"

if password == "REPLACE_WITH_YOUR_AACUSER_PASSWORD":
    raise ValueError("Enter your aacuser password.")

db = AnimalShelter(username, password)
db.client.admin.command("ping")


def read_animals(query):
    # Read records through the CRUD module.
    records = db.read(query)
    result = pd.DataFrame.from_records(records)
    if "name" in result.columns:
        result["name"] = result["name"].fillna("Not provided")
        result.loc[
            result["name"].astype(str).str.strip() == "",
            "name"
        ] = "Not provided"
    return result.drop(columns=["_id"], errors="ignore")


def table_records(frame):
    # Assign a stable ID to each record for row selection.
    rows = frame.to_dict("records")

    for index, row in enumerate(rows):
        record_number = row.get("rec_num")
        row["id"] = (
            str(record_number)
            if record_number is not None
            else str(index)
        )

    return rows


df = read_animals({})

if df.empty:
    raise RuntimeError("The database returned no records.")

print("Connected to MongoDB. Records:", len(df))


# Rescue requirements from the Dashboard Specifications Document.
rescue_rules = {
    "water": {
        "breeds": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ],
        "sex": "Intact Female",
        "age_min": 26,
        "age_max": 156
    },
    "mountain": {
        "breeds": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "age_min": 26,
        "age_max": 156
    },
    "disaster": {
        "breeds": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ],
        "sex": "Intact Male",
        "age_min": 20,
        "age_max": 300
    }
}


def rescue_query(rescue_type):
    if rescue_type in (None, "reset"):
        return {}

    rule = rescue_rules[rescue_type]
    breeds = list(rule["breeds"])

    # Include the Mix version when it exists in the dataset.
    for breed in rule["breeds"]:
        if not breed.endswith(" Mix"):
            breeds.append(breed + " Mix")

    return {
        "animal_type": "Dog",
        "breed": {"$in": breeds},
        "sex_upon_outcome": rule["sex"],
        "age_upon_outcome_in_weeks": {
            "$gte": rule["age_min"],
            "$lte": rule["age_max"]
        }
    }


#########################
# Dashboard Layout / View
#########################

logo_path = Path("Grazioso Salvare Logo.png")

if not logo_path.exists():
    logo_path = Path("code_files/Grazioso Salvare Logo.png")

logo_data = base64.b64encode(
    logo_path.read_bytes()
).decode("utf-8")


# Use readable column headings for the data table.
column_labels = {
    "rec_num": "Record #",
    "animal_id": "Animal ID",
    "name": "Animal Name",
    "animal_type": "Animal Type",
    "breed": "Breed",
    "sex_upon_outcome": "Sex",
    "age_upon_outcome_in_weeks": "Age (weeks)",
    "outcome_type": "Outcome",
    "location_lat": "Latitude",
    "location_long": "Longitude"
}


columns = []

for field, label in column_labels.items():
    if field not in df.columns:
        continue

    column = {
        "name": label,
        "id": field,
        "selectable": True
    }

    # Display ages with two decimal places.
    if field == "age_upon_outcome_in_weeks":
        column["type"] = "numeric"
        column["format"] = {"specifier": ".2f"}

    # Keep coordinates readable without losing map precision.
    elif field in ("location_lat", "location_long"):
        column["type"] = "numeric"
        column["format"] = {"specifier": ".6f"}

    columns.append(column)


app = JupyterDash(__name__)

app.layout = html.Div([

    # Logo and dashboard heading.
    html.Div([
        html.A(
            html.Img(
                src="data:image/png;base64," + logo_data,
                style={
                    "height": "95px",
                    "objectFit": "contain"
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        html.Div([
            html.H1(
                "Grazioso Salvare Rescue Dashboard",
                style={
                    "marginBottom": "5px",
                    "color": "#203a63"
                }
            ),

            html.P(
                "Eden Valentinov | CS 340 Project Two",
                style={"marginTop": "0"}
            )
        ])

    ], style={
        "display": "flex",
        "gap": "22px",
        "alignItems": "center"
    }),

    html.Hr(),

    html.H3("Choose a rescue type"),

    # Rescue filter controls.
    dcc.RadioItems(
        id="filter-type",
        options=[
            {
                "label": "All / Reset",
                "value": "reset"
            },
            {
                "label": "Water Rescue",
                "value": "water"
            },
            {
                "label": "Mountain or Wilderness",
                "value": "mountain"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "disaster"
            }
        ],
        value="reset",
        labelStyle={
            "display": "inline-block",
            "marginRight": "23px",
            "marginBottom": "10px"
        }
    ),

    html.P(
        id="record-count",
        style={"fontWeight": "bold"}
    ),

    html.P(
        "Select the circle next to an animal to show its location on the map.",
        style={"color": "#526274"}
    ),

    # Interactive animal table.
    dash_table.DataTable(
        id="datatable-id",
        columns=columns,
        data=table_records(df),

        page_size=10,
        page_action="native",
        sort_action="native",
        filter_action="native",

        row_selectable="single",
        selected_rows=[0],
        selected_row_ids=[table_records(df)[0]["id"]],
        active_cell=None,

        style_table={
            "overflowX": "auto",
            "border": "1px solid #d9e3ee"
        },

        style_header={
            "backgroundColor": "#294e81",
            "color": "white",
            "fontWeight": "bold",
            "padding": "9px",
            "textAlign": "left"
        },

        style_cell={
            "textAlign": "left",
            "padding": "8px",
            "minWidth": "100px",
            "maxWidth": "225px",
            "overflow": "hidden",
            "textOverflow": "ellipsis",
            "fontFamily": "Arial",
            "fontSize": "13px"
        },

        style_data_conditional=[
            {
                "if": {"row_index": "odd"},
                "backgroundColor": "#f6f9fd"
            },
            {
                "if": {"state": "selected"},
                "backgroundColor": "#cce4ff",
                "color": "#153350"
            }
        ]
    ),

    html.Hr(style={"marginTop": "22px"}),

    # Pie chart and geolocation map.
    html.Div([

        html.Div(
            id="graph-id",
            style={
                "flex": "1 1 400px",
                "minWidth": "310px"
            }
        ),

        html.Div([

            html.H3(
                "Selected Animal Location",
                style={"color": "#294e81"}
            ),

            html.P(
                id="selected-animal",
                style={"marginTop": "0"}
            ),

            html.Div(id="map-id")

        ], style={
            "flex": "1 1 430px",
            "minWidth": "310px"
        })

    ], style={
        "display": "flex",
        "gap": "18px",
        "flexWrap": "wrap"
    }),

    html.Hr(),

    # Additional chart showing animal ages.
    html.Div(id="age-chart-id")

], style={
    "fontFamily": "Arial",
    "backgroundColor": "#fff",
    "color": "#233044",
    "padding": "22px",
    "maxWidth": "1450px",
    "margin": "0 auto"
})


#############################################
# Interaction Between Components / Controller
#############################################

# Update the table when the user selects a rescue filter.
@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "selected_rows"),
        Output("datatable-id", "selected_row_ids"),
        Output("datatable-id", "active_cell"),
        Output("record-count", "children")
    ],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):

    matching = table_records(
        read_animals(rescue_query(filter_type))
    )

    # Select the first matching record automatically.
    first_rows = [0] if matching else []
    first_ids = [matching[0]["id"]] if matching else []

    return (
        matching,
        first_rows,
        first_ids,
        None,
        f"Rescue filter matches: {len(matching):,}"
    )


def visible_frame(view_data, current_rows):
    # Do not reuse the original dataset if no results are found.
    if view_data is None:
        return pd.DataFrame(current_rows or [])

    return pd.DataFrame(view_data)



# Update both charts using the table's current data.
@app.callback(
    [
        Output("graph-id", "children"),
        Output("age-chart-id", "children")
    ],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "data")
    ]
)
def update_graphs(view_data, current_rows):

    view = visible_frame(view_data, current_rows)

    if view.empty or "breed" not in view.columns:
        message = "No animals to chart for the current selection."
        return html.P(message), html.P(message)

    # Count the breeds currently shown in the table.
    counts = view["breed"].fillna("Unknown").value_counts()

    if len(counts) > 8:
        main = counts.iloc[:8].copy()
        main.loc["Other breeds"] = counts.iloc[8:].sum()
        counts = main

    # Breed distribution chart.
    pie = px.pie(
        names=counts.index.tolist(),
        values=counts.values.tolist(),
        title="Breed Distribution",
        hole=0.28
    )

    pie.update_layout(
        height=420,
        legend_title_text="Breed",
        margin=dict(t=60, l=15, r=15, b=15)
    )

    # Age distribution chart.
    ages = pd.to_numeric(
        view["age_upon_outcome_in_weeks"],
        errors="coerce"
    )

    ranges = pd.cut(
        ages,
        bins=[-1, 26, 52, 104, 156, float("inf")],
        labels=[
            "0–26 weeks",
            "27–52 weeks",
            "53–104 weeks",
            "105–156 weeks",
            "157+ weeks"
        ]
    )

    order = [
        "0–26 weeks",
        "27–52 weeks",
        "53–104 weeks",
        "105–156 weeks",
        "157+ weeks"
    ]

    age_counts = ranges.value_counts().reindex(
        order,
        fill_value=0
    )

    age_data = pd.DataFrame({
        "Age group": order,
        "Animals": age_counts.values
    })

    bars = px.bar(
        age_data,
        x="Age group",
        y="Animals",
        text="Animals",
        title="Age Distribution of Displayed Animals"
    )

    bars.update_layout(
        height=360,
        margin=dict(t=70, l=45, r=25, b=45)
    )

    bars.update_traces(
        marker_color="#416ea5",
        textposition="outside"
    )

    # Allow enough space for labels above the bars.
    max_count = age_data["Animals"].max()

    bars.update_yaxes(
        range=[0, max(1, max_count * 1.2)],
        title_text="Animals"
    )

    return (
        dcc.Graph(figure=pie),
        dcc.Graph(figure=bars)
    )


# Update the map when a table row is selected.
@app.callback(
    [
        Output("map-id", "children"),
        Output("selected-animal", "children")
    ],
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "data"),
        Input("datatable-id", "derived_virtual_selected_row_ids"),
        Input("datatable-id", "active_cell")
    ]
)
def update_map(view_data, current_rows, selected_ids, active_cell):

    view = visible_frame(view_data, current_rows)

    if view.empty:
        return (
            html.P("No animal locations match this selection."),
            "No animal selected"
        )

    needed = {"location_lat", "location_long"}

    if not needed.issubset(view.columns):
        return (
            html.P("Location coordinates are missing."),
            "No location available"
        )

    # Track selection by row ID so sorting does not change
    # which animal is shown on the map.
    selected_id = None

    if selected_ids:
        selected_id = str(selected_ids[0])

    elif active_cell and active_cell.get("row_id") is not None:
        selected_id = str(active_cell["row_id"])

    view = view.copy()

    view["location_lat"] = pd.to_numeric(
        view["location_lat"],
        errors="coerce"
    )

    view["location_long"] = pd.to_numeric(
        view["location_long"],
        errors="coerce"
    )

    valid = view.dropna(
        subset=["location_lat", "location_long"]
    )

    valid = valid[
        valid["location_lat"].between(-90, 90)
        & valid["location_long"].between(-180, 180)
    ]

    if valid.empty:
        return (
            html.P("No valid coordinates for these animals."),
            "No location available"
        )

    # Find the selected animal.
    if selected_id is not None:
        found = valid[
            valid["id"].astype(str) == selected_id
        ]
    else:
        found = valid.iloc[0:0]

    if selected_id is not None and found.empty:
        return (
            html.P("The selected animal has no valid location coordinates."),
            "Selected animal location unavailable"
        )

    if not found.empty:
        animal = found.iloc[0]
    else:
        animal = valid.iloc[0]

    latitude = float(animal["location_lat"])
    longitude = float(animal["location_long"])

    name = str(animal.get("name") or "Unnamed animal")
    breed = str(animal.get("breed") or "Unknown breed")
    animal_id = str(animal.get("animal_id") or "Unknown ID")

    label = f"{name} ({animal_id}) — {breed}"

    marker = dl.Marker(
        position=[latitude, longitude],
        children=[
            dl.Tooltip(label),
            dl.Popup([
                html.B(name),
                html.P(f"ID: {animal_id}"),
                html.P(breed)
            ])
        ]
    )

    return (
        dl.Map(
            style={
                "width": "100%",
                "height": "415px"
            },
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(),
                marker
            ]
        ),
        label
    )


# Run the dashboard in Codio.
app.run_server(mode="external", port=8051, debug=False)


Connected to MongoDB. Records: 10006


 * Running on http://127.0.0.1:8051/ (Press CTRL+C to quit)
127.0.0.1 - - [22/Oct/2026 04:56:48] "GET /_alive_2ed69777-ec55-4a07-bf98-20d6ce3d6673 HTTP/1.1" 200 -


Dash app running on https://zoomferrari-melodygriffin-3000.codio.io/proxy/8051/


127.0.0.1 - - [22/Oct/2026 04:56:50] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:50] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:51] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:51] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Oct/2026 04:56:51] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Oct/2026 04:56:52] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:53] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:54] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [22/Oct/2026 04:56:54] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Oct/2026 04:56:54] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
127.0.0.1 - - [22/Oct/2026 04:56:58] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - -